# Baes MPNet Breadth Robustness Check

This notebook reruns semantic breadth with Baes et al.'s `sentence-transformers/all-mpnet-base-v2` sentence encoder. It is a target-only robustness check for the main XL-LEXEME breadth analysis: ADHD and Autism are retained, comparator terms are deliberately excluded.


## Setup

The annual publication-year axis, substantive-core target frames, uncapped target contexts, mean pairwise cosine distance, document-level bootstrap, and compact trend models are kept fixed. The only substantive change is the encoder: a generic SentenceTransformers MPNet sentence embedding replaces XL-LEXEME target-token embeddings.


In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
FRAME_LABEL_PATH = PROJECT_ROOT / "data/processed/lsc/classification/lsc_target_context_frame_labels.csv"
INTERIM_DIR = PROJECT_ROOT / "data/interim/lsc/breadth_baes_mpnet"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/lsc/breadth/robustness_baes_mpnet"
for directory in [INTERIM_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MPNET_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
LOCAL_MODEL_PATH = PROJECT_ROOT / "data/external/models/all-mpnet-base-v2"
LOCAL_FILES_ONLY = True
USE_LOCAL_MODEL_PATH_IF_AVAILABLE = True
MODEL_SOURCE = str(LOCAL_MODEL_PATH) if USE_LOCAL_MODEL_PATH_IF_AVAILABLE and LOCAL_MODEL_PATH.exists() else MPNET_MODEL_NAME
DEVICE = "cpu"
ENCODE_BATCH_SIZE = 32
REBUILD_MPNET_EMBEDDINGS = False

TARGET_UNITS = ["ADHD", "Autism"]
EXPECTED_YEARS = list(range(2014, 2027))
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
MIN_CONTEXT_TOKENS = 5
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
BOOTSTRAP_REPETITIONS = 500
RANDOM_SEED = 123
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75
TOKEN_RE = re.compile(r"\b\w+\b")
BOOTSTRAP_RNG = np.random.default_rng(RANDOM_SEED)

assert CONTEXT_PATH.exists(), CONTEXT_PATH
assert FRAME_LABEL_PATH.exists(), FRAME_LABEL_PATH


/opt/anaconda3/envs/msc-nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Target Contexts And Frame Labels

Baseline terms are not included in this robustness notebook. ADHD and Autism contexts are expanded into an Overall substantive-core stratum plus clinical, lived-experience, and mixed frame strata.


In [2]:
context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "collapsed_matched_texts",
    "registered_domain",
    "target_sentence",
    "target_sentence_plus_adjacent",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
contexts = contexts.loc[contexts["analysis_unit"].isin(TARGET_UNITS)].copy()
contexts["source_context_row_id"] = contexts.index.astype(int)
contexts["registered_domain"] = contexts["registered_domain"].fillna("unknown_domain")


def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]


frame_labels = pd.read_csv(
    FRAME_LABEL_PATH,
    usecols=["context_id", "predicted_derived_frame", "p_substantive", "p_clinical_given_substantive", "p_lived_given_substantive"],
)
if frame_labels["context_id"].duplicated().any():
    raise RuntimeError("Frame-label handoff contains duplicate context IDs.")

contexts["context_id"] = contexts.apply(stable_context_id, axis=1)
contexts = contexts.merge(frame_labels, on="context_id", how="left")
missing_target_labels = contexts["predicted_derived_frame"].isna().sum()
if missing_target_labels:
    raise RuntimeError(f"Missing frame labels for {missing_target_labels:,} target contexts.")

core_target_contexts = contexts.loc[contexts["predicted_derived_frame"].isin(CORE_TARGET_FRAMES)].copy()
target_by_frame = core_target_contexts.copy()
target_by_frame["frame_stratum"] = target_by_frame["predicted_derived_frame"]
target_overall = core_target_contexts.copy()
target_overall["frame_stratum"] = "substantive_core_overall"
analysis_contexts = pd.concat([target_overall, target_by_frame], ignore_index=True, sort=False)
analysis_contexts["context_row_id"] = np.arange(len(analysis_contexts), dtype=int)

observed_units = sorted(analysis_contexts["analysis_unit"].dropna().unique())
observed_years = sorted(analysis_contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(TARGET_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected target units: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years: {missing_years}")

input_summary = pd.DataFrame(
    {
        "metric": ["target_source_contexts", "analysis_context_rows", "documents", "analysis_units", "years", "model_source"],
        "value": [
            len(contexts),
            len(analysis_contexts),
            analysis_contexts["doc_id"].nunique(),
            ", ".join(observed_units),
            f"{min(observed_years)}-{max(observed_years)}",
            MODEL_SOURCE,
        ],
    }
)
input_summary


,metric,value
0,target_source_contexts,96864
1,analysis_context_rows,114224
2,documents,32709
3,analysis_units,"ADHD, Autism"
4,years,2014-2026
5,model_source,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/external/models/all-mpnet-base-v2


## Select Sentence Contexts

The same sentence-selection rule as the main breadth notebook is used, but the selected text is encoded as natural text without target markers because MPNet is a generic sentence encoder.


In [3]:
def token_count(text: object) -> int:
    return len(TOKEN_RE.findall(str(text or "")))


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def whitespace_flexible_pattern(text: str) -> str:
    escaped = re.escape(" ".join(str(text).split()))
    return escaped.replace(r"\ ", r"\s+")


def raw_form_patterns(raw_form: str) -> list[str]:
    patterns = {
        "adhd": [r"\bADHD\b"],
        "attention_deficit": [r"\battention\s+deficit(?:\s+hyperactivity(?:\s+disorder)?)?\b"],
        "autism": [r"\bautism\b"],
        "autistic": [r"\bautistic\b"],
        "autism_spectrum": [r"\bautism\s+spectrum\b"],
        "asd_disambiguated": [r"\bASD\b"],
    }
    return patterns.get(raw_form, [r"\b" + re.escape(raw_form.replace("_", " ")) + r"\b"])


def candidate_patterns(row: pd.Series) -> list[tuple[str, str]]:
    candidates: list[tuple[str, str]] = []
    matched_values = [row.get("matched_text")] + split_pipe_values(row.get("collapsed_matched_texts"))
    for value in matched_values:
        if isinstance(value, str) and value.strip():
            candidates.append(("matched_text", whitespace_flexible_pattern(value)))
    for pattern in raw_form_patterns(str(row.get("raw_form") or "")):
        candidates.append(("raw_form", pattern))
    seen: set[str] = set()
    unique_candidates = []
    for source, pattern in candidates:
        if pattern not in seen:
            unique_candidates.append((source, pattern))
            seen.add(pattern)
    return unique_candidates


def find_first_match(text: str, row: pd.Series) -> tuple[str | None, str | None]:
    for pattern_source, pattern in candidate_patterns(row):
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return pattern_source, text[match.start() : match.end()]
    return None, None


def context_candidates(row: pd.Series) -> list[tuple[str, str]]:
    sentence = str(row.get("target_sentence") or "").strip()
    adjacent = str(row.get("target_sentence_plus_adjacent") or "").strip()
    candidates: list[tuple[str, str]] = []
    if token_count(sentence) >= MIN_CONTEXT_TOKENS:
        candidates.append(("target_sentence", sentence))
        if adjacent and adjacent != sentence:
            candidates.append(("target_sentence_plus_adjacent", adjacent))
    else:
        if adjacent:
            candidates.append(("target_sentence_plus_adjacent", adjacent))
        if sentence:
            candidates.append(("target_sentence", sentence))
    return candidates


def select_embedding_context(row: pd.Series) -> dict[str, object]:
    for context_source, text in context_candidates(row):
        pattern_source, marked_text = find_first_match(text, row)
        if marked_text:
            return {
                "embedding_text": text,
                "context_source": context_source,
                "mark_pattern_source": pattern_source,
                "marked_text": marked_text,
                "context_token_count": token_count(text),
                "markable": True,
                "unmarkable_reason": "",
            }
    return {
        "embedding_text": "",
        "context_source": "",
        "mark_pattern_source": "",
        "marked_text": "",
        "context_token_count": 0,
        "markable": False,
        "unmarkable_reason": "target_span_not_found_in_context",
    }


mark_records = [
    select_embedding_context(row)
    for _, row in tqdm(analysis_contexts.iterrows(), total=len(analysis_contexts), desc="Selecting MPNet contexts")
]
mark_data = pd.DataFrame(mark_records)
selected_contexts = pd.concat([analysis_contexts, mark_data], axis=1)

unmarkable_contexts = selected_contexts.loc[~selected_contexts["markable"]].copy()
markable_contexts = selected_contexts.loc[selected_contexts["markable"]].copy()

dedupe_subset = ["analysis_unit", "lsc_year", "frame_stratum", "doc_id", "embedding_text"]
markable_before_dedupe = len(markable_contexts)
markable_contexts = markable_contexts.drop_duplicates(subset=dedupe_subset).reset_index(drop=True)
duplicate_contexts_removed = markable_before_dedupe - len(markable_contexts)

unmarkable_path = INTERIM_DIR / "lsc_baes_mpnet_unmarkable_contexts.csv"
unmarkable_columns = [
    "context_row_id",
    "source_context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "raw_form",
    "matched_text",
    "unmarkable_reason",
]
unmarkable_contexts[unmarkable_columns].to_csv(unmarkable_path, index=False)

pd.DataFrame(
    {
        "metric": ["markable_contexts", "unmarkable_contexts", "duplicate_contexts_removed"],
        "value": [len(markable_contexts), len(unmarkable_contexts), duplicate_contexts_removed],
    }
)


Selecting MPNet contexts: 100%|██████████| 114224/114224 [00:04<00:00, 27900.76it/s]


,metric,value
0,markable_contexts,104173
1,unmarkable_contexts,0
2,duplicate_contexts_removed,10051


## Keep All Target Contexts

No comparator sampling is needed. Every markable ADHD/Autism target context enters the robustness run, with diagnostics saved for raw-form balance and domain concentration.


In [4]:
GROUP_COLUMNS = ["analysis_unit", "lsc_year", "frame_stratum"]

sampled_contexts = markable_contexts.copy().reset_index(drop=True)
sampled_contexts["sample_row_id"] = np.arange(len(sampled_contexts), dtype=int)

available_counts = analysis_contexts.groupby(GROUP_COLUMNS, as_index=False).agg(available_contexts=("doc_id", "size"))
markable_counts = selected_contexts.loc[selected_contexts["markable"]].groupby(GROUP_COLUMNS, as_index=False).agg(markable_contexts_before_dedupe=("doc_id", "size"))
deduped_counts = markable_contexts.groupby(GROUP_COLUMNS, as_index=False).agg(markable_contexts=("doc_id", "size"))
sampled_counts = (
    sampled_contexts.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        sampled_contexts=("doc_id", "size"),
        sampled_documents=("doc_id", "nunique"),
        sampled_domains=("registered_domain", "nunique"),
        target_sentence_contexts=("context_source", lambda values: int((values == "target_sentence").sum())),
        adjacent_contexts=("context_source", lambda values: int((values == "target_sentence_plus_adjacent").sum())),
    )
)

top_domain_share = (
    sampled_contexts.groupby([*GROUP_COLUMNS, "registered_domain"], as_index=False)
    .size()
    .rename(columns={"size": "domain_contexts"})
)
top_domain_share["sampled_contexts_for_share"] = top_domain_share.groupby(GROUP_COLUMNS)["domain_contexts"].transform("sum")
top_domain_share["domain_share"] = top_domain_share["domain_contexts"] / top_domain_share["sampled_contexts_for_share"]
top_domain_share = top_domain_share.sort_values("domain_share", ascending=False).groupby(GROUP_COLUMNS, as_index=False).head(1)
top_domain_share = top_domain_share.rename(columns={"registered_domain": "top_domain", "domain_share": "top_domain_share"})[
    [*GROUP_COLUMNS, "top_domain", "top_domain_share"]
]

sampling_diagnostics = available_counts.merge(markable_counts, on=GROUP_COLUMNS, how="left")
sampling_diagnostics = sampling_diagnostics.merge(deduped_counts, on=GROUP_COLUMNS, how="left")
sampling_diagnostics = sampling_diagnostics.merge(sampled_counts, on=GROUP_COLUMNS, how="left")
sampling_diagnostics = sampling_diagnostics.merge(top_domain_share, on=GROUP_COLUMNS, how="left")
for column in ["markable_contexts_before_dedupe", "markable_contexts", "sampled_contexts", "sampled_documents", "sampled_domains"]:
    sampling_diagnostics[column] = sampling_diagnostics[column].fillna(0).astype(int)
sampling_diagnostics["unmarkable_contexts"] = sampling_diagnostics["available_contexts"] - sampling_diagnostics["markable_contexts_before_dedupe"]
sampling_diagnostics["duplicate_contexts_removed"] = sampling_diagnostics["markable_contexts_before_dedupe"] - sampling_diagnostics["markable_contexts"]
sampling_diagnostics["unmarkable_share"] = sampling_diagnostics["unmarkable_contexts"] / sampling_diagnostics["available_contexts"].replace(0, np.nan)
sampling_diagnostics["cap_applied"] = False
sampling_diagnostics["sampling_policy"] = "all_markable_target_contexts_no_comparators"
sampling_diagnostics["small_cell_flag"] = sampling_diagnostics["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    sampling_diagnostics["sampled_contexts"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | sampling_diagnostics["sampled_documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)
sampling_diagnostics = sampling_diagnostics.sort_values(GROUP_COLUMNS).reset_index(drop=True)

raw_form_diagnostics = (
    sampled_contexts.groupby([*GROUP_COLUMNS, "term_role", "target_group", "raw_form"], as_index=False)
    .agg(sampled_contexts=("doc_id", "size"), sampled_documents=("doc_id", "nunique"))
)
raw_form_diagnostics["raw_form_context_share"] = raw_form_diagnostics["sampled_contexts"] / raw_form_diagnostics.groupby(GROUP_COLUMNS)["sampled_contexts"].transform("sum")

sampled_contexts_path = INTERIM_DIR / "lsc_baes_mpnet_sampled_contexts.parquet"
sampling_diagnostics_path = PROCESSED_DIR / "lsc_baes_mpnet_sampling_diagnostics.csv"
raw_form_diagnostics_path = PROCESSED_DIR / "lsc_baes_mpnet_raw_form_diagnostics.csv"
sampling_diagnostics.to_csv(sampling_diagnostics_path, index=False)
raw_form_diagnostics.to_csv(raw_form_diagnostics_path, index=False)

sampling_diagnostics.head(12)


,analysis_unit,lsc_year,frame_stratum,available_contexts,markable_contexts_before_dedupe,markable_contexts,sampled_contexts,sampled_documents,sampled_domains,target_sentence_contexts,adjacent_contexts,top_domain,top_domain_share,unmarkable_contexts,duplicate_contexts_removed,unmarkable_share,cap_applied,sampling_policy,small_cell_flag
0,ADHD,2014,clinical_only,1286,1286,1210,1210,808,663,1207,3,naturalnews.com,0.020661,0,76,0.0,False,all_markable_target_contexts_no_comparators,False
1,ADHD,2014,lived_only,310,310,295,295,243,209,294,1,additudemag.com,0.071186,0,15,0.0,False,all_markable_target_contexts_no_comparators,False
2,ADHD,2014,mixed,183,183,173,173,152,133,173,0,rrstar.com,0.034682,0,10,0.0,False,all_markable_target_contexts_no_comparators,False
3,ADHD,2014,substantive_core_overall,1779,1779,1678,1678,1093,874,1674,4,additudemag.com,0.023838,0,101,0.0,False,all_markable_target_contexts_no_comparators,False
4,ADHD,2015,clinical_only,1203,1203,1124,1124,774,586,1119,5,rightdiagnosis.com,0.025801,0,79,0.0,False,all_markable_target_contexts_no_comparators,False
5,ADHD,2015,lived_only,216,216,204,204,169,151,204,0,flinger.us,0.039216,0,12,0.0,False,all_markable_target_contexts_no_comparators,False
6,ADHD,2015,mixed,128,128,118,118,109,99,117,1,flinger.us,0.050847,0,10,0.0,False,all_markable_target_contexts_no_comparators,False
7,ADHD,2015,substantive_core_overall,1547,1547,1446,1446,993,756,1440,6,rightdiagnosis.com,0.020055,0,101,0.0,False,all_markable_target_contexts_no_comparators,False
8,ADHD,2016,clinical_only,1149,1149,1087,1087,790,662,1076,11,wellesley.edu,0.023919,0,62,0.0,False,all_markable_target_contexts_no_comparators,False
9,ADHD,2016,lived_only,309,309,294,294,245,229,293,1,additudemag.com,0.023810,0,15,0.0,False,all_markable_target_contexts_no_comparators,False


## Offline Model Preflight

Run this cell before the long embedding pass. It checks whether `sentence-transformers/all-mpnet-base-v2` can be loaded with `local_files_only=True`, either from the local model folder or from the Hugging Face cache.


In [5]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError("Install sentence-transformers in the msc-nlp environment before running this robustness check.") from exc

print(f"Model source: {MODEL_SOURCE}")
print(f"Local files only: {LOCAL_FILES_ONLY}")
print("This preflight loads the model before the full embedding pass so offline/cache problems fail early.")
model = SentenceTransformer(MODEL_SOURCE, device=DEVICE, local_files_only=LOCAL_FILES_ONLY)
probe = model.encode(
    [
        "ADHD support needs are discussed in the classroom context.",
        "Autism diagnosis and lived experience appear in public discourse.",
    ],
    batch_size=2,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)
if probe.ndim != 2 or probe.shape[0] != 2:
    raise RuntimeError("MPNet preflight did not return the expected embedding matrix.")
print(f"Preflight embedding dimensions: {probe.shape[1]}")
print(f"Model max sequence length: {getattr(model, 'max_seq_length', 'unknown')}")


Model source: /Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/external/models/all-mpnet-base-v2
Local files only: True
This preflight loads the model before the full embedding pass so offline/cache problems fail early.
Preflight embedding dimensions: 768
Model max sequence length: 384


## Encode MPNet Contexts

This is the long-running cell. It writes reusable embedding caches under `data/interim/lsc/breadth_baes_mpnet/`; reruns load the cache unless `REBUILD_MPNET_EMBEDDINGS` is set to `True`.


In [6]:
embedding_path = INTERIM_DIR / "lsc_baes_mpnet_embeddings.npy"
embedding_normalised_path = INTERIM_DIR / "lsc_baes_mpnet_embeddings_normalised.npy"
embedding_index_path = INTERIM_DIR / "lsc_baes_mpnet_embedding_index.csv"
unique_contexts_path = INTERIM_DIR / "lsc_baes_mpnet_unique_contexts.parquet"
embedding_manifest_path = INTERIM_DIR / "lsc_baes_mpnet_embedding_manifest.json"

unique_contexts = sampled_contexts[["embedding_text"]].drop_duplicates().reset_index(drop=True)
unique_contexts["embedding_row_id"] = np.arange(len(unique_contexts), dtype=int)
current_contexts = unique_contexts["embedding_text"].tolist()


def fingerprint_contexts(contexts: list[str]) -> str:
    digest = hashlib.sha256()
    for context in contexts:
        digest.update(context.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def build_embedding_manifest(contexts: list[str], embedding_dimensions: int | None = None) -> dict[str, object]:
    return {
        "cache_schema_version": 1,
        "context_fingerprint": fingerprint_contexts(contexts),
        "unique_embedding_rows": len(contexts),
        "embedding_dimensions": embedding_dimensions,
        "model_name": MPNET_MODEL_NAME,
        "model_source": MODEL_SOURCE,
        "local_files_only": LOCAL_FILES_ONLY,
        "backend": "sentence-transformers",
    }


def manifest_matches_current_cache(manifest: dict[str, object], expected_manifest: dict[str, object]) -> bool:
    checked_keys = [
        "cache_schema_version",
        "context_fingerprint",
        "unique_embedding_rows",
        "model_name",
        "model_source",
        "local_files_only",
        "backend",
    ]
    return all(manifest.get(key) == expected_manifest.get(key) for key in checked_keys)


def validate_loaded_embeddings(
    loaded_embeddings: np.ndarray,
    loaded_embeddings_normalised: np.ndarray,
    loaded_unique_contexts: pd.DataFrame,
) -> None:
    expected_rows = len(loaded_unique_contexts)
    if loaded_embeddings.ndim != 2 or loaded_embeddings_normalised.ndim != 2:
        raise RuntimeError("Cached MPNet embeddings must be two-dimensional arrays.")
    if loaded_embeddings.shape != loaded_embeddings_normalised.shape:
        raise RuntimeError("Cached raw and normalised MPNet embeddings have different shapes.")
    if loaded_embeddings.shape[0] != expected_rows:
        raise RuntimeError("Cached MPNet embedding row count does not match unique contexts.")
    if not loaded_unique_contexts["embedding_row_id"].equals(pd.Series(np.arange(expected_rows), name="embedding_row_id")):
        raise RuntimeError("Cached MPNet embedding row IDs are not contiguous from zero.")
    if loaded_unique_contexts["embedding_text"].tolist() != current_contexts:
        raise RuntimeError("Cached MPNet contexts do not match the current context order.")
    normalised_norms = np.linalg.norm(loaded_embeddings_normalised, axis=1)
    if not np.all(np.isfinite(loaded_embeddings)) or not np.all(np.isfinite(loaded_embeddings_normalised)):
        raise RuntimeError("Cached MPNet embeddings contain non-finite values.")
    if not np.allclose(normalised_norms, 1.0, atol=1e-4):
        raise RuntimeError("Cached normalised MPNet embeddings are not unit length.")


def load_embedding_cache() -> tuple[np.ndarray, np.ndarray, pd.DataFrame, str] | None:
    if not embedding_path.exists() or not embedding_normalised_path.exists() or not embedding_manifest_path.exists() or not unique_contexts_path.exists():
        return None
    expected_manifest = build_embedding_manifest(current_contexts)
    manifest = json.loads(embedding_manifest_path.read_text())
    if not manifest_matches_current_cache(manifest, expected_manifest):
        return None
    cached_unique_contexts = pd.read_parquet(unique_contexts_path)
    loaded_embeddings = np.load(embedding_path)
    loaded_embeddings_normalised = np.load(embedding_normalised_path)
    cached_unique_contexts["embedding_row_id"] = cached_unique_contexts["embedding_row_id"].astype(int)
    validate_loaded_embeddings(loaded_embeddings, loaded_embeddings_normalised, cached_unique_contexts)
    return loaded_embeddings, loaded_embeddings_normalised, cached_unique_contexts, "loaded_manifest_cache"


loaded_cache = None if REBUILD_MPNET_EMBEDDINGS else load_embedding_cache()
if loaded_cache is not None:
    embeddings, embeddings_normalised, unique_contexts, embedding_cache_status = loaded_cache
else:
    # Reuse the model loaded in the preflight cell if this notebook is run top-to-bottom.
    if "model" not in globals():
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(MODEL_SOURCE, device=DEVICE, local_files_only=LOCAL_FILES_ONLY)
    embeddings = model.encode(
        current_contexts,
        batch_size=ENCODE_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=True,
    ).astype("float32")
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    if np.any(norms == 0) or not np.all(np.isfinite(norms)):
        raise RuntimeError("MPNet produced zero or non-finite embedding norms.")
    embeddings_normalised = (embeddings / norms).astype("float32")
    embedding_cache_status = "rebuilt_embeddings"
    np.save(embedding_path, embeddings)
    np.save(embedding_normalised_path, embeddings_normalised)

unique_contexts.to_parquet(unique_contexts_path, index=False)
embedding_manifest = build_embedding_manifest(current_contexts, embedding_dimensions=int(embeddings.shape[1]))
embedding_manifest_path.write_text(json.dumps(embedding_manifest, indent=2, sort_keys=True) + "\n")

sampled_contexts = sampled_contexts.merge(unique_contexts, on="embedding_text", how="left")
if sampled_contexts["embedding_row_id"].isna().any():
    raise RuntimeError("Some sampled contexts did not receive an embedding row ID.")
sampled_contexts["embedding_row_id"] = sampled_contexts["embedding_row_id"].astype(int)

embedding_index = sampled_contexts[
    [
        "sample_row_id",
        "embedding_row_id",
        "context_row_id",
        "source_context_row_id",
        "context_id",
        "doc_id",
        "lsc_year",
        "analysis_unit",
        "term_role",
        "target_group",
        "raw_form",
        "frame_stratum",
        "predicted_derived_frame",
        "registered_domain",
        "context_source",
        "context_token_count",
        "marked_text",
    ]
].copy()
embedding_index.to_csv(embedding_index_path, index=False)
sampled_contexts.to_parquet(sampled_contexts_path, index=False)

pd.DataFrame(
    {
        "metric": [
            "embedding_cache_status",
            "sampled_context_rows",
            "unique_embedding_rows",
            "embedding_dimensions",
            "model_source",
        ],
        "value": [
            embedding_cache_status,
            len(sampled_contexts),
            int(embeddings.shape[0]),
            int(embeddings.shape[1]),
            MODEL_SOURCE,
        ],
    }
)


Batches: 100%|██████████| 1463/1463 [16:25<00:00,  1.49it/s]


,metric,value
0,embedding_cache_status,rebuilt_embeddings
1,sampled_context_rows,104173
2,unique_embedding_rows,46791
3,embedding_dimensions,768
4,model_source,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/external/models/all-mpnet-base-v2


## Annual Breadth Scores

Breadth is the annual mean pairwise cosine distance among normalised sentence embeddings for each target/frame/year cell.


In [7]:
def mean_pairwise_cosine_distance(normalised_vectors: np.ndarray) -> float:
    n = normalised_vectors.shape[0]
    if n < 2:
        return float("nan")
    sum_vector = normalised_vectors.sum(axis=0)
    sum_pairwise_similarity = (float(np.dot(sum_vector, sum_vector)) - n) / 2.0
    pair_count = n * (n - 1) / 2.0
    mean_similarity = sum_pairwise_similarity / pair_count
    return float(1.0 - mean_similarity)


def bootstrap_document_breadth(group_index: pd.DataFrame, normalised_vectors: np.ndarray, rng: np.random.Generator) -> dict[str, object]:
    document_to_rows = [values.to_numpy(dtype=int) for _, values in group_index.groupby("doc_id")["embedding_row_id"]]
    if len(document_to_rows) < 2:
        return {
            "bootstrap_repetitions": 0,
            "bootstrap_unit": "doc_id",
            "breadth_bootstrap_mean": float("nan"),
            "breadth_ci_low": float("nan"),
            "breadth_ci_high": float("nan"),
        }
    bootstrap_values = []
    for _ in range(BOOTSTRAP_REPETITIONS):
        sampled_docs = rng.integers(0, len(document_to_rows), len(document_to_rows))
        sampled_rows = np.concatenate([document_to_rows[index] for index in sampled_docs])
        if len(sampled_rows) >= 2:
            bootstrap_values.append(mean_pairwise_cosine_distance(normalised_vectors[sampled_rows]))
    return {
        "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
        "bootstrap_unit": "doc_id",
        "breadth_bootstrap_mean": float(np.mean(bootstrap_values)),
        "breadth_ci_low": float(np.quantile(bootstrap_values, 0.025)),
        "breadth_ci_high": float(np.quantile(bootstrap_values, 0.975)),
    }


breadth_records = []
for (unit, year, frame_stratum), group in embedding_index.groupby(["analysis_unit", "lsc_year", "frame_stratum"], sort=True):
    row_ids = group["embedding_row_id"].to_numpy(dtype=int)
    vectors = embeddings_normalised[row_ids]
    metadata = group.iloc[0]
    bootstrap = bootstrap_document_breadth(group, embeddings_normalised, BOOTSTRAP_RNG)
    breadth_records.append(
        {
            "lsc_year": int(year),
            "analysis_unit": unit,
            "frame_stratum": frame_stratum,
            "term_role": metadata["term_role"],
            "target_group": metadata["target_group"],
            "breadth_mean_pairwise_cosine_distance": mean_pairwise_cosine_distance(vectors),
            "sampled_contexts": int(len(group)),
            "sampled_documents": int(group["doc_id"].nunique()),
            "sampled_domains": int(group["registered_domain"].nunique()),
            **bootstrap,
        }
    )

annual_breadth = pd.DataFrame(breadth_records).sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_breadth = annual_breadth.merge(
    sampling_diagnostics[
        [
            "analysis_unit",
            "lsc_year",
            "frame_stratum",
            "available_contexts",
            "markable_contexts",
            "unmarkable_contexts",
            "unmarkable_share",
            "cap_applied",
            "sampling_policy",
            "top_domain",
            "top_domain_share",
            "small_cell_flag",
        ]
    ],
    on=["analysis_unit", "lsc_year", "frame_stratum"],
    how="left",
)

annual_breadth_path = PROCESSED_DIR / "lsc_baes_mpnet_breadth_annual_scores.csv"
annual_breadth.to_csv(annual_breadth_path, index=False)
annual_breadth.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,breadth_mean_pairwise_cosine_distance,sampled_contexts,sampled_documents,sampled_domains,bootstrap_repetitions,bootstrap_unit,breadth_bootstrap_mean,breadth_ci_low,breadth_ci_high,available_contexts,markable_contexts,unmarkable_contexts,unmarkable_share,cap_applied,sampling_policy,top_domain,top_domain_share,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,0.617783,1210,808,663,500,doc_id,0.617546,0.607170,0.627502,1286,1210,0,0.0,False,all_markable_target_contexts_no_comparators,naturalnews.com,0.020661,False
1,2015,ADHD,clinical_only,target,ADHD,0.621315,1124,774,586,500,doc_id,0.620690,0.609793,0.630812,1203,1124,0,0.0,False,all_markable_target_contexts_no_comparators,rightdiagnosis.com,0.025801,False
2,2016,ADHD,clinical_only,target,ADHD,0.635684,1087,790,662,500,doc_id,0.635295,0.624481,0.645262,1149,1087,0,0.0,False,all_markable_target_contexts_no_comparators,wellesley.edu,0.023919,False
3,2017,ADHD,clinical_only,target,ADHD,0.626752,1038,751,655,500,doc_id,0.626540,0.617572,0.637004,1099,1038,0,0.0,False,all_markable_target_contexts_no_comparators,rehabs.com,0.018304,False
4,2018,ADHD,clinical_only,target,ADHD,0.631969,1123,787,719,500,doc_id,0.631607,0.621903,0.642157,1182,1123,0,0.0,False,all_markable_target_contexts_no_comparators,bedford.gov.uk,0.015138,False
5,2019,ADHD,clinical_only,target,ADHD,0.650587,946,674,619,500,doc_id,0.649592,0.638094,0.660401,1000,946,0,0.0,False,all_markable_target_contexts_no_comparators,flavio.pl,0.009514,False
6,2020,ADHD,clinical_only,target,ADHD,0.632642,913,668,622,500,doc_id,0.632014,0.622382,0.642284,959,913,0,0.0,False,all_markable_target_contexts_no_comparators,digication.com,0.006572,False
7,2021,ADHD,clinical_only,target,ADHD,0.634392,881,636,579,500,doc_id,0.633048,0.620940,0.643948,928,881,0,0.0,False,all_markable_target_contexts_no_comparators,suffolkfamilytherapy.com,0.012486,False
8,2022,ADHD,clinical_only,target,ADHD,0.634768,877,634,595,500,doc_id,0.633810,0.622912,0.644445,930,877,0,0.0,False,all_markable_target_contexts_no_comparators,thestudiolife.com,0.013683,False
9,2023,ADHD,clinical_only,target,ADHD,0.622807,715,477,446,500,doc_id,0.622133,0.609470,0.635256,750,715,0,0.0,False,all_markable_target_contexts_no_comparators,turningwinds.com,0.023776,False


## Trend Models And Audit Flags

Trend summaries mirror the main scalar LSC notebooks. A comparison table checks slope direction against the main XL-LEXEME target trends.


In [8]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
for group_values, frame in annual_breadth.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    trend_rows.append(
        {
            "analysis_unit": analysis_unit,
            "frame_stratum": frame_stratum,
            "term_role": term_role,
            "target_group": target_group,
            "index_name": "baes_mpnet_breadth_mean_pairwise_cosine_distance",
            **fit_trend(frame, "breadth_mean_pairwise_cosine_distance"),
        }
    )
trend_summary = pd.DataFrame(trend_rows)
trend_summary_path = PROCESSED_DIR / "lsc_baes_mpnet_breadth_trend_models.csv"
trend_summary.to_csv(trend_summary_path, index=False)

main_trend_path = PROJECT_ROOT / "data/processed/lsc/breadth/lsc_breadth_trend_models.csv"
if main_trend_path.exists():
    main_trends = pd.read_csv(main_trend_path)
    main_subset = main_trends.loc[
        main_trends["index_name"].eq("breadth_mean_pairwise_cosine_distance")
        & main_trends["analysis_unit"].isin(TARGET_UNITS)
    ].copy()
    comparison = trend_summary.merge(
        main_subset[
            [
                "analysis_unit",
                "frame_stratum",
                "linear_slope_per_year",
                "linear_p_value",
                "linear_adj_r_squared",
                "autocorrelation_flag",
            ]
        ].rename(
            columns={
                "linear_slope_per_year": "xl_lexeme_linear_slope_per_year",
                "linear_p_value": "xl_lexeme_linear_p_value",
                "linear_adj_r_squared": "xl_lexeme_linear_adj_r_squared",
                "autocorrelation_flag": "xl_lexeme_autocorrelation_flag",
            }
        ),
        on=["analysis_unit", "frame_stratum"],
        how="left",
    )
    comparison = comparison.rename(
        columns={
            "linear_slope_per_year": "baes_mpnet_linear_slope_per_year",
            "linear_p_value": "baes_mpnet_linear_p_value",
            "linear_adj_r_squared": "baes_mpnet_linear_adj_r_squared",
            "autocorrelation_flag": "baes_mpnet_autocorrelation_flag",
        }
    )
    comparison["slope_direction_agrees"] = np.sign(comparison["baes_mpnet_linear_slope_per_year"]) == np.sign(
        comparison["xl_lexeme_linear_slope_per_year"]
    )
    comparison_path = PROCESSED_DIR / "lsc_baes_mpnet_xl_lexeme_trend_comparison.csv"
    comparison.to_csv(comparison_path, index=False)
else:
    comparison = pd.DataFrame()

flag_rows = []
for row in annual_breadth.itertuples(index=False):
    flags = []
    if bool(row.small_cell_flag):
        flags.append("small_frame_year_cell")
    if row.sampled_contexts < 100:
        flags.append("sampled_contexts_lt_100")
    if row.sampled_documents < 50:
        flags.append("sampled_documents_lt_50")
    if row.unmarkable_share > 0.05:
        flags.append("unmarkable_share_gt_0_05")
    if pd.notna(row.top_domain_share) and row.top_domain_share > 0.35:
        flags.append("top_domain_share_gt_0_35")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "flags": ";".join(flags),
                "sampled_contexts": row.sampled_contexts,
                "sampled_documents": row.sampled_documents,
                "top_domain": row.top_domain,
                "top_domain_share": row.top_domain_share,
                "unmarkable_share": row.unmarkable_share,
            }
        )
audit_flags = pd.DataFrame(flag_rows)
audit_flags_path = PROCESSED_DIR / "lsc_baes_mpnet_breadth_audit_flags.csv"
audit_flags.to_csv(audit_flags_path, index=False)

trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.629315,-0.000019,0.000747,0.979905,0.000060,-0.090843,-0.007769,1.560998,0.162498,False,NaN,NaN,0.156931,0.247774
1,ADHD,lived_only,target,ADHD,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.617743,-0.002029,0.000748,0.020204,0.400775,0.346300,-0.633068,2.317035,-0.231018,False,NaN,NaN,0.343138,-0.003162
2,ADHD,mixed,target,ADHD,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.624649,-0.001941,0.001148,0.118952,0.206307,0.134153,-0.454210,3.360535,-0.692619,True,-0.002117,0.004087,0.053410,-0.080743
3,ADHD,substantive_core_overall,target,ADHD,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.635806,-0.000449,0.000568,0.445480,0.053861,-0.032151,-0.232080,1.646754,0.114662,False,NaN,NaN,0.269834,0.301985
4,Autism,clinical_only,target,Autism,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.634003,-0.000189,0.000598,0.758011,0.008989,-0.081103,-0.094810,0.784203,0.542138,True,-0.001834,0.053583,0.382971,0.464074
5,Autism,lived_only,target,Autism,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.611181,-0.002461,0.000968,0.027437,0.369847,0.312560,-0.608150,2.563227,-0.398880,False,NaN,NaN,0.251511,-0.061049
6,Autism,mixed,target,Autism,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.604348,-0.001404,0.000887,0.141611,0.185639,0.111606,-0.430858,2.288975,-0.207037,False,NaN,NaN,0.190150,0.078543
7,Autism,substantive_core_overall,target,Autism,baes_mpnet_breadth_mean_pairwise_cosine_distance,13,2020.0,0.635131,-0.001422,0.000335,0.001372,0.621180,0.586742,-0.788150,1.349812,0.150198,False,NaN,NaN,0.619919,0.033177


## Handoff Summary

This check fails if the expected target-only annual frame grid is incomplete.


In [9]:
expected_annual_rows = len(EXPECTED_YEARS) * len(TARGET_UNITS) * len(TARGET_FRAME_STRATA)
summary = {
    "annual_rows": len(annual_breadth),
    "expected_annual_rows": expected_annual_rows,
    "sampled_context_rows": len(sampled_contexts),
    "unique_embedding_rows": int(embeddings.shape[0]),
    "embedding_dimensions": int(embeddings.shape[1]),
    "trend_rows": len(trend_summary),
    "audit_flag_rows": len(audit_flags),
    "comparison_rows": len(comparison),
    "model_source": MODEL_SOURCE,
}
if summary["annual_rows"] != expected_annual_rows:
    raise RuntimeError(f"Expected {expected_annual_rows} annual rows, found {summary['annual_rows']}.")
summary


{'annual_rows': 104,
 'expected_annual_rows': 104,
 'sampled_context_rows': 104173,
 'unique_embedding_rows': 46791,
 'embedding_dimensions': 768,
 'trend_rows': 8,
 'audit_flag_rows': 1,
 'comparison_rows': 8,
 'model_source': '/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/external/models/all-mpnet-base-v2'}